# E-Commerce RAG Chatbot – ShopEasy Support Assistant

**Domain:** E-Commerce Customer Support  
**Framework:** LangChain · OpenAI · FAISS  
**Key Features:** Document ingestion · Semantic retrieval · Conversation memory · Strict grounding

---

## Notebook Flow
| Section | Description |
|---|---|
| 1 | Install & import dependencies |
| 2 | Ingest documents → FAISS index |
| 3 | Build the retriever |
| 4 | Build the LLM + prompt |
| 5 | Single-turn QA demo |
| 6 | Multi-turn conversation demo (with memory) |
| 7 | Fallback safety test |
| 8 | Retrieval inspection |
| 9 | Interactive chat loop |

## Section 1 – Install Dependencies

Install all required packages. Skip if already installed.

In [0]:
# Install required packages
%pip install -q openai langchain langchain-openai langchain-community faiss-cpu pypdf python-dotenv fpdf2 tiktoken

## Section 2 – Ingest Documents → FAISS Index

**Pipeline:**
1. Load all PDFs with `PyPDFLoader`
2. Split into overlapping chunks with `RecursiveCharacterTextSplitter`
3. Embed with `text-embedding-3-small`
4. Store in local FAISS index

**Chunking strategy:** `chunk_size=700`, `chunk_overlap=120` — chosen to keep policy clauses intact while avoiding context overflow.

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

DATA_FOLDER    = "E-Commerce_RAG_Chatbot/documents"
VECTOR_DB_PATH = "ecommerce_faiss_index"

# ── Load PDFs ──────────────────────────────────────────────
print("[1/4] Loading PDFs ...")
documents = []
for file in sorted(os.listdir(DATA_FOLDER)):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_FOLDER, file))
        pages  = loader.load()
        documents.extend(pages)
        print(f"   {file}: {len(pages)} pages")

print(f"\n   Total pages: {len(documents)}")

# ── Split ──────────────────────────────────────────────────
print("\n[2/4] Splitting into chunks ...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", "!", "?", " "],
    length_function=len,
)
chunks = splitter.split_documents(documents)
print(f"   Total chunks: {len(chunks)}")
print(f"\n   Sample chunk:\n{'-'*50}\n{chunks[0].page_content[:400]}\n{'-'*50}")

# ── Embed ──────────────────────────────────────────────────
print("\n[3/4] Generating embeddings ...")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.getenv("OPENAI_API_BASE"),
)

# ── Index ──────────────────────────────────────────────────
print("\n[4/4] Building FAISS index ...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(VECTOR_DB_PATH)

print(f"\n✅ FAISS index saved to '{VECTOR_DB_PATH}/'")
print("   Files:", os.listdir(VECTOR_DB_PATH))

[1/4] Loading PDFs ...
   return_policy.pdf: 2 pages
   shipping_policy.pdf: 2 pages
   smarthub_x1_manual.pdf: 2 pages
   warranty_policy.pdf: 2 pages

   Total pages: 8

[2/4] Splitting into chunks ...
   Total chunks: 25

   Sample chunk:
--------------------------------------------------
Return & Refund Policy
ShopEasy E-Commerce  |  Document Version 1.0
1. Overview
At ShopEasy, customer satisfaction is our top priority. We offer a transparent and hassle-free return and refund
process for all eligible purchases. This policy applies to all products sold directly through ShopEasy.com or the
ShopEasy mobile application.
2. Return Window
Customers may return most items within the foll
--------------------------------------------------

[3/4] Generating embeddings ...
/usr/local/lib/python3.10/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from crypt

## Section 3 – Build the Retriever

Load the FAISS index and create a retriever. We use `k=5` (top-5 chunks) to handle multi-document queries (e.g., return + warranty at the same time).

In [2]:
vectorstore = FAISS.load_local(
    VECTOR_DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

print("✅ Retriever ready (top-5 similarity search)")

✅ Retriever ready (top-5 similarity search)


## Section 4 – Build LLM, Prompt & Chain

**Design choices:**
- `temperature=0` — deterministic policy answers
- `MessagesPlaceholder` — plugs conversation history into every prompt
- System prompt explicitly instructs the LLM to **only answer from retrieved context** and use the exact fallback message when information is absent

In [3]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.getenv("OPENAI_API_BASE"),
)

SYSTEM_PROMPT = """You are a knowledgeable and friendly E-Commerce Support Assistant for ShopEasy.

Your responsibilities:
- Help customers with returns, shipping, warranty, product, account, and privacy queries.
- Answer ONLY using the information found in the retrieved document context.
- If the answer is not in the documents, respond with exactly:
  "I don't have enough information in the provided documents."
- Do NOT make up policies, numbers, or procedures not in the documents.
- Be concise, warm, and professional.
- Use bullet points for multi-step processes.
"""

HUMAN_TEMPLATE = """--- Retrieved Document Context ---
{context}
--- End of Context ---

Customer Question: {question}"""

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", HUMAN_TEMPLATE),
])


# ── Helper functions ──────────────────────────────────────
def get_context(question: str) -> str:
    docs = retriever.invoke(question)
    parts = []
    for i, doc in enumerate(docs, 1):
        src  = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", "?")
        parts.append(f"[Doc {i} | {src} | p.{page}]\n{doc.page_content}")
    return "\n\n".join(parts)


def ask(question: str, history: list = []) -> str:
    context  = get_context(question)
    messages = prompt.format_messages(context=context, question=question, history=history)
    return llm.invoke(messages).content


print("✅ LLM, prompt, and helper functions ready.")

✅ LLM, prompt, and helper functions ready.


## Section 5 – Single-Turn QA Demo

Test individual questions across all document types.

In [4]:
test_questions = [
    "What is the return window for electronics?",
    "How long does international shipping take?",
    "What is covered under the SmartHub X1 warranty?",
    "How do I set up the SmartHub X1?",
    "What are ShopEasy Coins and how do I use them?",
]

print("=" * 70)
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("-" * 70)


Q: What is the return window for electronics?
A: The return window for electronics is 15 days from the date of delivery.
----------------------------------------------------------------------

Q: How long does international shipping take?
A: International shipping takes approximately 7-20 business days, depending on the destination. Specifically, the International Standard shipping option takes 10-20 business days.
----------------------------------------------------------------------

Q: What is covered under the SmartHub X1 warranty?
A: The warranty for the SmartHub X1 covers:

- Manufacturing defects present at the time of purchase
- Hardware failures due to faulty components under normal use
- Display defects (dead pixels above threshold: more than 3 dead pixels)
- Battery capacity below 80% within the warranty period (if applicable)

If you have any further questions, feel free to ask!
----------------------------------------------------------------------

Q: How do I set up the 

## Section 6 – Multi-Turn Conversation Demo (with Memory)

Demonstrates **context-aware follow-up** questions. The chatbot uses conversation history so it can answer "What about eligibility?" without losing track of the previous topic.

In [5]:
def chat_turn(question: str, history: list) -> tuple[str, list]:
    """Run one turn and update history."""
    reply = ask(question, history)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=reply))
    return reply, history


# ── Conversation 1: Returns ───────────────────────────────
print("\n" + "="*70)
print("CONVERSATION 1 – Return & Refund")
print("="*70)

history1 = []

q1 = "What is the return policy for electronics?"
a1, history1 = chat_turn(q1, history1)
print(f"\nYou: {q1}\nBot: {a1}")
print("-"*70)

q2 = "What if I received a defective one?"
a2, history1 = chat_turn(q2, history1)
print(f"\nYou: {q2}\nBot: {a2}")
print("-"*70)

q3 = "How long does the refund take?"
a3, history1 = chat_turn(q3, history1)
print(f"\nYou: {q3}\nBot: {a3}")
print("-"*70)


CONVERSATION 1 – Return & Refund

You: What is the return policy for electronics?
Bot: The return policy for electronics at ShopEasy is as follows:

- Return window: 15 days from the date of delivery.
- Eligibility criteria:
  - Item must be in its original, unused condition.
  - All original packaging, accessories, and documentation must be included.
  - Return request must be initiated within the 15-day return window.
  - A valid proof of purchase (order ID or receipt) must be provided.
- If you receive a damaged or defective electronic product, contact us within 48 hours of delivery for a free pick-up and either a replacement or a full refund.
----------------------------------------------------------------------

You: What if I received a defective one?
Bot: If you receive a defective product, please follow these steps:

- Contact us within 48 hours of delivery.
- Attach clear photographs of the damage or defect along with your order ID when raising the complaint.
- We will arrang

In [6]:
# ── Conversation 2: Product Manual ───────────────────────
print("\n" + "="*70)
print("CONVERSATION 2 – SmartHub X1 Product Manual")
print("="*70)

history2 = []

q1 = "How do I set up the SmartHub X1?"
a1, history2 = chat_turn(q1, history2)
print(f"\nYou: {q1}\nBot: {a1}")
print("-"*70)

q2 = "What do the LED colours mean?"
a2, history2 = chat_turn(q2, history2)
print(f"\nYou: {q2}\nBot: {a2}")
print("-"*70)

q3 = "What about its warranty?"
a3, history2 = chat_turn(q3, history2)
print(f"\nYou: {q3}\nBot: {a3}")
print("-"*70)


CONVERSATION 2 – SmartHub X1 Product Manual

You: How do I set up the SmartHub X1?
Bot: To set up the SmartHub X1, follow these steps:

1. **Connect to Router**: Use the provided Ethernet cable to connect the SmartHub X1 to your router.
2. **Power On**: Plug in the power adapter. The LED ring will glow red during boot (30-60 seconds).
3. **Download App**: Download the 'ShopEasy Home' app from the App Store or Google Play.
4. **Open App**: Open the app, tap '+', and select 'SmartHub X1'.
5. **Pairing**: Follow the in-app pairing wizard. The LED ring will turn blue when connected.
6. **Create Home**: Create your home and add rooms to start pairing smart devices.
----------------------------------------------------------------------

You: What do the LED colours mean?
Bot: The LED status indicators for the SmartHub X1 are as follows:

- **Solid Red**: Booting up or no internet connection
- **Blinking Red**: Firmware update in progress (do not unplug)
- **Solid Blue**: Connected and opera

## Section 7 – Fallback Safety Test

If the answer is **not** in the knowledge base, the bot must respond with exactly:
> *"I don't have enough information in the provided documents."*

Test with an out-of-scope question.

In [7]:
out_of_scope_questions = [
    "What is the capital of France?",
    "Can you write Python code for a sorting algorithm?",
    "What is ShopEasy's stock price?",
]

print("FALLBACK SAFETY TESTS")
print("="*70)
for q in out_of_scope_questions:
    answer = ask(q)
    fallback_triggered = "don't have enough information" in answer.lower()
    status = "✅ FALLBACK" if fallback_triggered else "⚠️  ANSWERED (review needed)"
    print(f"\nQ: {q}")
    print(f"A: {answer}")
    print(f"Status: {status}")
    print("-"*70)

FALLBACK SAFETY TESTS

Q: What is the capital of France?
A: I don't have enough information in the provided documents.
Status: ✅ FALLBACK
----------------------------------------------------------------------

Q: Can you write Python code for a sorting algorithm?
A: I don't have enough information in the provided documents.
Status: ✅ FALLBACK
----------------------------------------------------------------------

Q: What is ShopEasy's stock price?
A: I don't have enough information in the provided documents.
Status: ✅ FALLBACK
----------------------------------------------------------------------


## Section 8 – Retrieval Inspection

Inspect which document chunks are retrieved for a given query. This helps understand what the LLM is working from.

In [8]:
def inspect_retrieval(question: str):
    print(f"Query: {question}")
    print("="*70)
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs, 1):
        src  = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", "?")
        print(f"\n[Chunk {i}] Source: {src} | Page: {page}")
        print("-"*50)
        print(doc.page_content[:300] + ("..." if len(doc.page_content) > 300 else ""))
    print()


inspect_retrieval("What happens if I receive a damaged product?")
inspect_retrieval("How do I reset the SmartHub X1 to factory settings?")

Query: What happens if I receive a damaged product?

[Chunk 1] Source: return_policy.pdf | Page: 1
--------------------------------------------------
If you receive a damaged or defective product, please contact us within 48 hours of delivery. Attach clear
photographs of the damage/defect along with your order ID when raising the complaint. We will arrange a free
pick-up and either replace the item or issue a full refund -- your choice.
7. Partia...

[Chunk 2] Source: warranty_policy.pdf | Page: 0
--------------------------------------------------
4. What Is NOT Covered
The warranty does NOT cover:
  - Physical damage caused by the customer (drops, cracks, liquid damage)
  - Damage caused by unauthorised repair or modification
  - Normal wear and tear (scratches, fading of paint/colour)
  - Damage caused by misuse, negligence, or accidents
  ...

[Chunk 3] Source: warranty_policy.pdf | Page: 0
--------------------------------------------------
Step 3 - Our team will review the claim wi

## Section 9 – Interactive Chat Loop

Full interactive chatbot with:
- Rolling conversation memory (last 10 turns)
- `clear` to reset history
- `exit` to quit

> **Note:** Run this cell in a terminal / script for best experience. In Jupyter, input() works but will block the kernel.

In [9]:
MAX_HISTORY = 10

def trim_history(history, max_turns=MAX_HISTORY):
    cutoff = max_turns * 2
    return history[-cutoff:] if len(history) > cutoff else history


def run_chatbot():
    print("\n" + "="*65)
    print("  ShopEasy Customer Support Chatbot")
    print("  Type 'exit' to quit | 'clear' to reset conversation")
    print("="*65 + "\n")

    history = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue
        if user_input.lower() == "exit":
            print("\nThank you for using ShopEasy Support. Goodbye! 👋")
            break
        if user_input.lower() == "clear":
            history.clear()
            print("[Conversation history cleared.]\n")
            continue

        reply = ask(user_input, history)
        print(f"\nShopEasy Bot: {reply}\n" + "-"*65)

        history.append(HumanMessage(content=user_input))
        history.append(AIMessage(content=reply))
        history = trim_history(history)


run_chatbot()


  ShopEasy Customer Support Chatbot
  Type 'exit' to quit | 'clear' to reset conversation


ShopEasy Bot: To install the ShopEasy SmartHub X1, please follow these steps:

1. **Connect to Router**: 
   - Use the provided Ethernet cable to connect the SmartHub X1 to your router.

2. **Power Up**: 
   - Plug in the 12V/2A power adapter. The LED ring will glow red during boot (this may take 30-60 seconds).

3. **Download the App**: 
   - Download the 'ShopEasy Home' app from the App Store or Google Play.

4. **Open the App**: 
   - Open the app, tap '+', and select 'SmartHub X1'.

5. **Follow Pairing Wizard**: 
   - Follow the in-app pairing wizard. The LED ring will turn blue when connected.

6. **Create Home and Add Rooms**: 
   - Create your home and add rooms to start pairing smart devices.

If you have any further questions, feel free to ask!
-----------------------------------------------------------------

ShopEasy Bot: The ShopEasy SmartHub X1 is a versatile smart home controller 